In [33]:
import pandas as pd 
from scipy import stats
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/Dataset_spine.csv")
df

,pelvic_incidence,pelvic_tilt,lumbar_lordosis_angle,sacral_slope,pelvic_radius,degree_spondylolisthesis,pelvic_slope,Direct_tilt,thoracic_slope,cervical_tilt,sacrum_angle,scoliosis_slope,Class_att
0,63.027817,22.552586,39.609117,40.475232,98.672917,-0.254400,0.744503,12.5661,14.5386,15.30468,-28.658501,43.5123,Abnormal
1,39.056951,10.060991,25.015378,28.995960,114.405425,4.564259,0.415186,12.8874,17.5323,16.78486,-25.530607,16.1102,Abnormal
2,68.832021,22.218482,50.092194,46.613539,105.985135,-3.530317,0.474889,26.8343,17.4861,16.65897,-29.031888,19.2221,Abnormal
3,69.297008,24.652878,44.311238,44.644130,101.868495,11.211523,0.369345,23.5603,12.7074,11.42447,-30.470246,18.8329,Abnormal
4,49.712859,9.652075,28.317406,40.060784,108.168725,7.918501,0.543360,35.4940,15.9546,8.87237,-16.378376,24.9171,Abnormal
...,...,...,...,...,...,...,...,...,...,...,...,...,...
305,47.903565,13.616688,36.000000,34.286877,117.449062,-4.245395,0.129744,7.8433,14.7484,8.51707,-15.728927,11.5472,Normal
306,53.936748,20.721496,29.220534,33.215251,114.365845,-0.421010,0.047913,19.1986,18.1972,7.08745,6.013843,43.8693,Normal
307,61.446597,22.694968,46.170347,38.751628,125.670725,-2.707880,0.081070,16.2059,13.5565,8.89572,3.564463,18.4151,Normal
308,45.252792,8.693157,41.583126,36.559635,118.545842,0.214750,0.159251,14.7334,16.0928,9.75922,5.767308,33.7192,Normal


بخش 1:

select data (انتخاب داده)

In [34]:
df.groupby('Class_att').mean(numeric_only= True)

,pelvic_incidence,pelvic_tilt,lumbar_lordosis_angle,sacral_slope,pelvic_radius,degree_spondylolisthesis,pelvic_slope,Direct_tilt,thoracic_slope,cervical_tilt,sacrum_angle,scoliosis_slope
Class_att,,,,,,,,,,,,
Abnormal,64.692562,19.791111,55.925370,44.90145,115.077713,37.777705,0.483979,21.085875,12.948913,12.132737,-13.826677,25.146915
Normal,51.685244,12.821414,43.542605,38.86383,123.890834,2.186572,0.449880,21.816394,13.307268,11.514534,-14.528712,26.694019


تفاوت میانگین بخش 1 داده ها خیلی بیشتر از بخش 2 هستش که باز میشه به توزیع تصادفیشون شک کرد

چون میانگین هاشون خیلی بهم نزدیکه میتونن دقت مدل رو پایین تر هم بیارن و از نظر من بهتره حذف بشن

In [35]:
columns_to_check = ["pelvic_slope", "Direct_tilt",
                    "thoracic_slope", "cervical_tilt",
                    "sacrum_angle", "scoliosis_slope"]
for column in columns_to_check:
    normal = df[df["Class_att"] == "Normal"][column]
    abnormal = df[df["Class_att"] == "Abnormal"][column]
    t_stat, p_value = stats.ttest_ind(normal,abnormal)
    print(column,"p_value: ",round(p_value, 4))

pelvic_slope p_value:  0.3269
Direct_tilt p_value:  0.4874
thoracic_slope p_value:  0.3865
cervical_tilt p_value:  0.0786
sacrum_angle p_value:  0.6372
scoliosis_slope p_value:  0.2236


اگر مقدار زیر 0.05 باشه معنی داره ولی هیچکدوم زیر این مقدار نیست پس بهتره حذف بشن

In [36]:
selected_df = df.drop(columns= columns_to_check)
selected_df.shape

(310, 7)

بخش 2:

clean data (پاکسازی داده)

In [37]:
selected_df[selected_df["degree_spondylolisthesis"] > 140]


,pelvic_incidence,pelvic_tilt,lumbar_lordosis_angle,sacral_slope,pelvic_radius,degree_spondylolisthesis,Class_att
75,70.221452,39.822724,68.118403,30.398728,148.525562,145.378143,Abnormal
95,57.522356,33.647075,50.909858,23.875281,140.981712,148.753711,Abnormal
115,129.834041,8.404475,48.384057,121.429566,107.690466,418.543082,Abnormal


داده 115 با بقیه رکورد ها تو ستون degree_spondylolisthesis متفاوت هستش و

 با توجه به این دوتا رکورد دیگه من 2 تا راه دارم 
 
 یک اینکه این رکورد مشکوک رو حذف کنم ولی چون دیتاستم کلا310 رکورد داره

 و بقیه فیلد های رکورد عادیه منقی نیست 

 دو اینکه از capping استفاده کنم و اثیر مخربش رو کم کنم 

 منطقی ترین کار در حال حاضر برای من capping هستش



In [38]:
Q1= selected_df["degree_spondylolisthesis"].quantile(0.25)
Q3= selected_df["degree_spondylolisthesis"].quantile(0.75)

IQR= Q3 - Q1

upper_bound = Q3 + 1.5 + IQR
print(upper_bound)

82.47097725125


مرز بالامون 82.47 هستش که خوب فاصلش با 418 خیلی زیاده  

In [39]:
selected_df.loc[selected_df["degree_spondylolisthesis"]>200,"degree_spondylolisthesis" ] = upper_bound

selected_df[selected_df["degree_spondylolisthesis"]>82]

,pelvic_incidence,pelvic_tilt,lumbar_lordosis_angle,sacral_slope,pelvic_radius,degree_spondylolisthesis,Class_att
61,89.680567,32.704435,83.130732,56.976132,129.955476,92.027277,Abnormal
71,86.900794,32.928168,47.794347,53.972627,135.075364,101.719092,Abnormal
75,70.221452,39.822724,68.118403,30.398728,148.525562,145.378143,Abnormal
76,86.753609,36.043016,69.221045,50.710593,139.414504,110.860782,Abnormal
95,57.522356,33.647075,50.909858,23.875281,140.981712,148.753711,Abnormal
104,77.409333,29.396545,63.232302,48.012788,118.450731,93.563737,Abnormal
114,80.988074,36.843172,86.960602,44.144903,141.088149,85.872152,Abnormal
115,129.834041,8.404475,48.384057,121.429566,107.690466,82.470977,Abnormal
135,77.121344,30.349874,77.481083,46.771470,110.611148,82.093607,Abnormal
141,89.504947,48.903653,72.003423,40.601295,134.634291,118.353370,Abnormal


In [40]:
selected_df["degree_spondylolisthesis"].max()

np.float64(148.7537109)

با دستور بالا فقط داده 418 رو با بالاترین مرزمون عوض کردم  بقیه اوتلایر ها رو دست نزدم 

و الان بزرگترین مقدار این مستون 148.75 هستش بجای 418


In [41]:
(selected_df.select_dtypes(include="number")<0).sum()

pelvic_incidence             0
pelvic_tilt                  8
lumbar_lordosis_angle        0
sacral_slope                 0
pelvic_radius                0
degree_spondylolisthesis    58
dtype: int64

چون از نظر پزشکی میتونه زاویه ها به چپ و راست باشه 

پس امکان داره شیب راست عدد مثبت در نظر گرقته شده باشه و شیب چپ عدد منفی 

به خاطر همین موضوع من دست به اعداد منفی دیتاستم نمیزنم

بخش 3:

construct data(ساخت ویژگی جدید)

ستون pelvice_tilt وsacral_slope همبستگی خیلی زیادی داشتند (0.81)

pelvic_incidence و احتمالا جمع اون وتاست از نظر پزشکی باید اول چکش کنیم :


In [42]:
selected_df["sum_check"] = selected_df["sacral_slope"] + selected_df["pelvic_tilt"]

selected_df[["pelvic_incidence" , "sum_check" ]].head(15)

,pelvic_incidence,sum_check
0,63.027817,63.027817
1,39.056951,39.056951
2,68.832021,68.832021
3,69.297008,69.297008
4,49.712859,49.712859
5,40.250200,40.250200
6,53.432928,53.432928
7,45.366754,45.366754
8,43.790190,43.790190
9,36.686353,36.686353


دقیقا همینجوری بود پس باید یکی از فیلد رو حذف کنیم که ایده ال ترینش pelvic_incidence هستش و از جمع 2 فیلد دیگه بدست میاد 

In [43]:
selected_df = selected_df.drop(columns=["pelvic_incidence", "sum_check"])
selected_df

,pelvic_tilt,lumbar_lordosis_angle,sacral_slope,pelvic_radius,degree_spondylolisthesis,Class_att
0,22.552586,39.609117,40.475232,98.672917,-0.254400,Abnormal
1,10.060991,25.015378,28.995960,114.405425,4.564259,Abnormal
2,22.218482,50.092194,46.613539,105.985135,-3.530317,Abnormal
3,24.652878,44.311238,44.644130,101.868495,11.211523,Abnormal
4,9.652075,28.317406,40.060784,108.168725,7.918501,Abnormal
...,...,...,...,...,...,...
305,13.616688,36.000000,34.286877,117.449062,-4.245395,Normal
306,20.721496,29.220534,33.215251,114.365845,-0.421010,Normal
307,22.694968,46.170347,38.751628,125.670725,-2.707880,Normal
308,8.693157,41.583126,36.559635,118.545842,0.214750,Normal


بخش 4:

integrate data(یکپارچه سازی داده ها )

چون فقط همین یک دیتا ست رو دارم این بخش چیزی برای انجام دادن نداره برام

بخش 5:

formate data (قالب بندی نهایی)

In [44]:
selected_df["Class_att"] = selected_df["Class_att"].map({"Normal" : 0 , "Abnormal" : 1 })
selected_df["Class_att"].value_counts()

Class_att
1    210
0    100
Name: count, dtype: int64

برای مدلسازی بهتره فیلد هدف باینری باشه و من اینجا همینکارو کردم 

0 به معنی نرمال بودن

1 به معنی انرمال بودن

In [47]:
X = selected_df.drop(columns = ["Class_att"])
Y = selected_df["Class_att"] 

print(X.shape)
print(Y.shape)

(310, 5)
(310,)


In [49]:
x_train , x_test ,y_train ,y_test = train_test_split(X, Y ,test_size = 0.2 ,random_state = 42 , stratify =Y)

print(x_train.shape)
print(x_test.shape)

(248, 5)
(62, 5)


تا اینجا تونستم دادههام رو برای مدل سازی اماده کنم 

248 رکورد برای train کردن 

62 رکورد برای testکردن

In [50]:
x_train.to_csv("x_train.csv",index= False)
x_test.to_csv("x_test.csv",index=False)
y_train.to_csv("y_train.csv",index=False)
y_test.to_csv("y_test.csv",index=False)